## 由于各个SV工具的vcf结果文件里总是将大部分SV的类型以BND表示，对于SV类型的统计很不友好，因此需要对vcf结果文件进行BND转换。

## （此pipeline紧接sv工具结果）

# 一、需要用到的脚本

### reclassify_bnd_by_tool.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import gzip
import re
import shutil
from collections import Counter
from pathlib import Path


BND_RE = re.compile(r"([\[\]])([^:\[\]]+):([0-9]+)([\[\]])")
ALT_ORIENTATION_TOOLS = {"lumpy", "manta", "delly", "svaba"}
MATE_PAIR_TOOLS = {"gridss", "gripss"}
NO_CONVERT_TOOLS = {"cue"}


def open_text(path, mode):
    if str(path).endswith(".gz"):
        return gzip.open(path, mode + "t")
    return open(path, mode)


def parse_info(info):
    d = {}
    if info in {"", "."}:
        return d
    for item in info.split(";"):
        if not item:
            continue
        if "=" in item:
            k, v = item.split("=", 1)
            d[k] = v
        else:
            d[item] = True
    return d


def format_info(info):
    out = []
    for k, v in info.items():
        if v is True:
            out.append(k)
        elif v is False or v is None:
            continue
        else:
            out.append(f"{k}={v}")
    return ";".join(out) if out else "."


def infer_tool(path):
    s = str(path).lower()
    for tool in ["gridss", "gripss", "lumpy", "manta", "delly", "svaba", "cue"]:
        if tool in s:
            return tool
    return "unknown"


def norm_chrom(chrom):
    chrom = str(chrom or "")
    return chrom[3:] if chrom.lower().startswith("chr") else chrom


def is_bnd_record(fields):
    if len(fields) < 8:
        return False
    info = parse_info(fields[7])
    svtype = str(info.get("SVTYPE", "")).upper()
    return svtype == "BND" or bool(BND_RE.search(fields[4]))


def parse_bnd_alt(alt):
    m = BND_RE.search(alt)
    if not m:
        return None
    bracket, rchrom, rpos, _ = m.groups()
    local_side = "5" if alt[0] in "[]" else "3"
    remote_side = "5" if bracket == "[" else "3"
    return {
        "remote_chrom": rchrom,
        "remote_pos": int(rpos),
        "local_side": local_side,
        "remote_side": remote_side,
    }


def literal_insert_len(ref, alt):
    seq = BND_RE.sub("", alt)
    seq = "".join(x for x in seq.upper() if x in "ACGTN")
    return max(0, len(seq) - len(ref))


def classify_breakend(chrom, pos, ref, alt, ins_max_distance, min_inserted_seq):
    bnd = parse_bnd_alt(alt)
    if not bnd:
        return "BND", "unparsed_BND_ALT"

    rchrom = bnd["remote_chrom"]
    rpos = bnd["remote_pos"]
    if norm_chrom(chrom) != norm_chrom(rchrom):
        return "TRA", "different_chromosome"

    ins_len = literal_insert_len(ref, alt)
    if abs(pos - rpos) <= ins_max_distance and ins_len >= min_inserted_seq:
        return "INS", f"short_distance_with_inserted_sequence_len={ins_len}"

    if pos <= rpos:
        left_side = bnd["local_side"]
        right_side = bnd["remote_side"]
    else:
        left_side = bnd["remote_side"]
        right_side = bnd["local_side"]

    pair = left_side + "to" + right_side
    if pair == "3to5":
        return "DEL", pair
    if pair == "5to3":
        return "DUP", pair
    if pair in {"3to3", "5to5"}:
        return "INV", pair
    return "BND", "unresolved_orientation"


def record_id(fields):
    return fields[2] if len(fields) > 2 else ""


def mate_ids(info):
    value = info.get("MATEID")
    if not value or value is True:
        return []
    return [x for x in str(value).split(",") if x]


def update_bnd_info(fields, new_type, method):
    info = parse_info(fields[7])
    orig = str(info.get("SVTYPE", "BND")).upper() or "BND"
    info["ORIG_SVTYPE"] = orig
    info["BND_RECLASS"] = new_type
    info["BND_RECLASS_METHOD"] = method.replace(";", ",")
    info["SVTYPE"] = new_type
    fields[7] = format_info(info)
    return fields


def collect_vcf_files(inputs, include_checkpoints):
    files = []
    for item in inputs:
        p = Path(item)
        if p.is_dir():
            files.extend(sorted(p.rglob("*.vcf")))
            files.extend(sorted(p.rglob("*.vcf.gz")))
        elif p.exists():
            files.append(p)
    out = []
    for p in files:
        if not include_checkpoints and ".ipynb_checkpoints" in p.parts:
            continue
        out.append(p)
    return out


def output_path_for(infile, outdir, tool):
    name = infile.name
    if name.endswith(".vcf.gz"):
        out_name = name[:-7] + ".bnd_reclass.vcf.gz"
    elif name.endswith(".vcf"):
        out_name = name[:-4] + ".bnd_reclass.vcf"
    else:
        out_name = name + ".bnd_reclass.vcf"
    return outdir / tool / out_name


def read_records(path):
    headers = []
    records = []
    with open_text(path, "r") as handle:
        for line in handle:
            if line.startswith("#"):
                headers.append(line)
            else:
                records.append(line.rstrip("\n").split("\t"))
    return headers, records


def write_records(path, headers, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open_text(path, "w") as out:
        inserted = False
        for line in headers:
            if line.startswith("#CHROM") and not inserted:
                out.write('##INFO=<ID=ORIG_SVTYPE,Number=1,Type=String,Description="Original SVTYPE before BND reclassification">\n')
                out.write('##INFO=<ID=BND_RECLASS,Number=1,Type=String,Description="SVTYPE inferred from BND breakend representation">\n')
                out.write('##INFO=<ID=BND_RECLASS_METHOD,Number=1,Type=String,Description="Rule used for BND reclassification">\n')
                inserted = True
            out.write(line)
        for fields in records:
            out.write("\t".join(fields) + "\n")


def process_vcf(infile, outdir, ins_max_distance, min_inserted_seq):
    tool = infer_tool(infile)
    outfile = output_path_for(infile, outdir, tool)
    headers, records = read_records(infile)
    counts = Counter()

    id_to_record = {record_id(fields): fields for fields in records if record_id(fields)}

    converted = []
    for fields in records:
        if len(fields) < 8:
            converted.append(fields)
            counts["malformed"] += 1
            continue

        if not is_bnd_record(fields):
            converted.append(fields)
            counts["non_bnd"] += 1
            continue

        chrom = fields[0]
        try:
            pos = int(float(fields[1]))
        except Exception:
            pos = 0
        ref = fields[3]
        alt = fields[4]
        info = parse_info(fields[7])

        if tool in NO_CONVERT_TOOLS:
            converted.append(fields)
            counts["bnd_kept_tool_no_convert"] += 1
            continue

        if tool in MATE_PAIR_TOOLS:
            mids = mate_ids(info)
            mate_ok = False
            for mid in mids:
                mate = id_to_record.get(mid)
                if mate and is_bnd_record(mate) and parse_bnd_alt(alt) and parse_bnd_alt(mate[4]):
                    mate_ok = True
                    break
            if not mate_ok:
                converted.append(update_bnd_info(fields, "BND", "gridss_no_reliable_mate_pair"))
                counts["bnd_kept_no_reliable_mate_pair"] += 1
                continue

        if tool in ALT_ORIENTATION_TOOLS or tool in MATE_PAIR_TOOLS or tool == "unknown":
            new_type, method = classify_breakend(chrom, pos, ref, alt, ins_max_distance, min_inserted_seq)
            converted.append(update_bnd_info(fields, new_type, method))
            counts[f"bnd_to_{new_type}"] += 1
        else:
            converted.append(fields)
            counts["bnd_kept_unknown_tool"] += 1

    write_records(outfile, headers, converted)
    return tool, infile, outfile, counts


def main():
    parser = argparse.ArgumentParser(
        description="Tool-aware conservative BND reclassification for CUE/Delly/Lumpy/Manta/SvABA/GRIDSS VCF files."
    )
    parser.add_argument("-i", "--input", nargs="+", required=True, help="VCF/VCF.gz files or directories.")
    parser.add_argument("-o", "--outdir", required=True, help="Output directory.")
    parser.add_argument("--include-checkpoints", action="store_true", help="Also process .ipynb_checkpoints files.")
    parser.add_argument("--ins-max-distance", type=int, default=50)
    parser.add_argument("--min-inserted-seq", type=int, default=20)
    parser.add_argument("--summary-name", default="bnd_reclass_by_tool.summary.tsv")
    args = parser.parse_args()

    outdir = Path(args.outdir)
    files = collect_vcf_files(args.input, args.include_checkpoints)
    if not files:
        raise SystemExit("No VCF/VCF.gz files found.")

    results = []
    for infile in files:
        results.append(process_vcf(infile, outdir, args.ins_max_distance, args.min_inserted_seq))

    summary = outdir / args.summary_name
    summary.parent.mkdir(parents=True, exist_ok=True)
    with open(summary, "wt") as out:
        out.write("tool\tinput_vcf\toutput_vcf\tmetric\tcount\n")
        for tool, infile, outfile, counts in results:
            for metric, count in sorted(counts.items()):
                out.write(f"{tool}\t{infile}\t{outfile}\t{metric}\t{count}\n")

    print(f"[DONE] processed_vcfs={len(results)}")
    print(f"[OUT] {outdir}")
    print(f"[SUMMARY] {summary}")


if __name__ == "__main__":
    main()


逻辑：
cue:
  不转换

lumpy / manta / delly / svaba:
  标准 BND ALT 方向可解析时：
    跨染色体 -> TRA
    同染色体 3to5 -> DEL
    同染色体 5to3 -> DUP
    同染色体 3to3 / 5to5 -> INV
    短距离且有明显插入序列 -> INS

gridss / gripss:
  必须有 MATEID
  mate 记录必须存在
  两条 breakend ALT 都能解析
  满足这些才转换
  否则保留 BND

# 二、处理步骤

### 运行脚本：

In [ ]:
chmod +x reclassify_bnd_by_tool.py

python reclassify_bnd_by_tool.py \
  -i /mnt/home/ygjx/chenkejin/share_group_folder_ygjx/Pancreatic_datasets/PDAC_WGS/sv_tools_results/ \
  -o /mnt/home/ygjx/chenkejin/sv_tools_results_bnd_reclass

### 汇总每个工具的转换结果：

In [ ]:
awk -F'\t' '
NR>1{
  count[$1"\t"$4]+=$5
}
END{
  for(k in count) print k"\t"count[k]
}' /mnt/home/ygjx/chenkejin/sv_tools_results_bnd_reclass/bnd_reclass_by_tool.summary.tsv \
| sort